# Stock Price Prediction Models

#### Tasks to complete:

- Add full introduction when project is complete
- Research walk-forward validation
- Look into Pesaran-Timmermann test
- Look into block bootstrap
- Consider adding macroeconomic data from FRED (federal reserve economic data)
- Consider whether to incorporate Alpha Vantage data as well - leaning towards no for now

#### Immediate next steps:

- Decide on whther to use alphabetical selection by sector or name recognition.
- Check chosen stocks are sufficiently decorrelated
- Make a plan for data analysis section

## Contents

1. [Introduction](#introduction)
2. [Setup](#setup)
3. [Data Collection](#data-collection)
4. [Data Cleaning](#data-cleaning)
5. [Data Analysis](#data-analysis)
6. [Model Fitting](#model-fitting)
7. [Model Evaluation](#model-evaluation)
8. [Conclusion](#conclusion)

## Introduction

Predicting stock prices is a notoriously challenging problem but one of great interest. Many have attempted to do so with mixed results. The primary reason for the difficulty lies in the efficient market hypothesis; the idea that any given stock's price reflects its true value and no discrepancies exist. If this is true, it would suggest there is no information that can be gleaned from historical data. There are many arguments to be made for and against this but regardless of its validity, it is clear that markets are very efficient, even if not fully, and this makes predicting stocks challenging as there is typically a very low signal to noise ratio in historical stock data. Other issues that make this can include the limitations of the dataset used; certain datasets may reflect patterns that were only present at a given moment in history.

**Project Aim:** The purpose of this project is predict stock prices given past historical data. We will measure whether the prediction correctly predicts the direction in which a stock will move. Knowing this with any chance greater than 50% will ultimately provide an advantage, although the magnitude of such an advantage will depend on several variables which we will discuss later. We will compare several predictive models and assess our level of confidence in these predictions by conducting a hypothesis test using walk-forward validation.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import requests
from io import StringIO

## Data Collection

The first task is choosing which stocks to analyse. We will use several criteria:
- Sufficient historic data
- Sufficiently liquid stocks
- Moderately volatile stocks
- Weakly correlated and from different sectors

For the first criteria, we'll only consider stocks that have data from 1990 to present day. This time period covers several key financial events: the dotcom bubble (2000), the financial crash (2008), the COVID-19 pandemic (2020) and others and will help test how well are predictions cope under rare and drastic events. We want to examine stocks with high liquidity as these stocks typically have cleaner price data. Moderate volatility makes the prediction interesting, stocks with little volatility are easier to predict but high volatility stocks likely have a worse signal to noise ratio. We'll also examine stocks from different sectors, this should help to ensure they are weakly correlated.

To help narrow down our search, we'll examine stocks from the S&P500. Such stocks have a long history and are highly liquid, thus fulfilling the first two criteria. We'll scrape the data from Wikipedia and filter companies by annualised volatility.

In [19]:
# Scrape Wikipedia page for S&P 500 table

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
html = requests.get(url, headers={'User-Agent':'Mozilla/5.0'}).text
df = pd.read_html(StringIO(html))[0]

# Download S&P 500 stocks from 1990 to 2026

tickers = list(df['Symbol'])
sp500_stocks = yf.download(tickers, start='1990-01-01', end='2026-01-01')

[************          26%                       ]  133 of 503 completed$BF.B: possibly delisted; no price data found  (1d 1990-01-01 -> 2026-01-01)
[****************      33%                       ]  165 of 503 completed$ECHO: possibly delisted; no price data found  (1d 1990-01-01 -> 2026-01-01)
[**********************47%                       ]  235 of 503 completed$FDXF: possibly delisted; no price data found  (1d 1990-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 631170000, endDate = 1767243600")
[**********************60%****                   ]  300 of 503 completed$BRK.B: possibly delisted; no timezone found
[*********************100%***********************]  502 of 503 completed

4 Failed downloads:
['BF.B', 'ECHO']: possibly delisted; no price data found  (1d 1990-01-01 -> 2026-01-01)
['FDXF']: possibly delisted; no price data found  (1d 1990-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 631170000, endDate = 1767243600")
['BR

In [96]:
# Filter by close price and drop any stocks which don't have complete data from 1990
# Only 235 stocks remain after this

close_prices = sp500_stocks.loc[:,'Close'].dropna(axis=1)

# Now compute annualised volatility

annualised_volatility = close_prices.pct_change().std()*np.sqrt(252)

# Filter by stocks with a volatility between 20 and 35%
# This gives us 151 stocks

moderate_volatility_stocks = close_prices.loc[:, (annualised_volatility<=0.35) & (annualised_volatility>=0.2)]

# Check what sectors are most frequently appearing

sectors = df[df['Symbol'].isin(moderate_volatility_stocks)].groupby('GICS Sector').count().sort_values(by='Symbol', ascending=False)
sectors.head()

,Symbol,Security,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
GICS Sector,,,,,,,
Industrials,39,39,39,39,39,39,39
Consumer Staples,22,22,22,22,22,22,22
Financials,20,20,20,20,20,20,20
Utilities,19,19,19,19,19,19,19
Health Care,18,18,18,18,18,18,18


Industrials, consumer staples and and financials are the biggest sectors shortly followed by utilities. These sectors all behave in fundamentally different ways during regime changes, making them fairly excellent candidates for our analysis. Utilities is a close 4th, but is quite similar to consumer staples in that it typically performs well during economic downturn due to its inherent necessity.

In [103]:
df[df['GICS Sector'].isin(sectors.index[0:3] ) & df['Symbol'].isin(moderate_volatility_stocks)].sort_values(by='GICS Sector').drop_duplicates('GICS Sector')

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
132,COST,Costco,Consumer Staples,Consumer Staples Merchandise Retail,"Issaquah, Washington",1993-10-01,909832,1976
402,SPGI,S&P Global,Financials,Financial Exchanges & Data,"New York City, New York",1957-03-04,64040,1917
363,PNR,Pentair,Industrials,Industrial Machinery & Supplies & Components,"Worsley, United Kingdom",2012-10-01,77360,1966


## Data Cleaning

## Data Analysis

## Model Fitting

## Model Evaluation

## Conclusion